In [ ]:
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
from empyrical import (max_drawdown, alpha_beta, annual_volatility,
                        sharpe_ratio, sortino_ratio)


## Performance Metrics

Rolling Sharpe, monthly statistics, and equity curves for both strategies.

In [ ]:
def rolling_sharpe(returns, window=63, annualization=252):
    \"\"\"Rolling Sharpe ratio over a sliding window (default ~1 quarter).\"\"\"
    roll = returns.rolling(window)
    mean = roll.mean()
    std  = roll.std(ddof=1)
    return (mean / std) * np.sqrt(annualization)


def rolling_drawdown(returns, window=63):
    \"\"\"Rolling maximum drawdown over a sliding window.\"\"\"
    cum = (1 + returns).cumprod()
    rolling_max = cum.rolling(window, min_periods=1).max()
    drawdown = (cum - rolling_max) / rolling_max
    return drawdown.rolling(window).min()


def summarise_strategy(label, returns, benchmark, annualization=252):
    \"\"\"
    Print and return a full performance summary including:
    - Overall metrics (drawdown, alpha, beta, vol, Sharpe, Sortino)
    - Monthly return stats
    - Rolling Sharpe plots
    \"\"\"
    returns  = pd.Series(returns).dropna()
    benchmark = pd.Series(benchmark).dropna()

    # ── Overall metrics ──────────────────────────────────────────────────
    maxdd  = max_drawdown(returns)
    alpha, beta_val = alpha_beta(returns, benchmark)
    ann_vol = annual_volatility(returns, annualization=annualization)
    sharpe  = sharpe_ratio(returns, annualization=annualization)
    sortino = sortino_ratio(returns, annualization=annualization)

    print(f"\n{'='*55}")
    print(f"  Performance Metrics — {label}")
    print(f"{'='*55}")
    print(f"  Maximum Drawdown    : {maxdd:.4%}")
    print(f"  Alpha               : {alpha:.4%}")
    print(f"  Beta                : {beta_val:.4f}")
    print(f"  Annual Volatility   : {ann_vol:.4f}")
    print(f"  Sharpe Ratio        : {sharpe:.4f}")
    print(f"  Sortino Ratio       : {sortino:.4f}")

    # ── Monthly return breakdown ──────────────────────────────────────────
    if hasattr(returns.index, 'to_period'):
        monthly = (1 + returns).resample('ME').prod() - 1
    else:
        returns.index = pd.to_datetime(returns.index)
        monthly = (1 + returns).resample('ME').prod() - 1

    print(f"\n  Monthly Return Stats:")
    print(f"    Mean   : {monthly.mean():.4%}")
    print(f"    Median : {monthly.median():.4%}")
    print(f"    Std    : {monthly.std():.4%}")
    print(f"    Min    : {monthly.min():.4%}")
    print(f"    Max    : {monthly.max():.4%}")
    print(f"    % Positive months: {(monthly > 0).mean():.1%}")

    # ── Rolling Sharpe plot ───────────────────────────────────────────────
    roll_sharpe = rolling_sharpe(returns, window=63)
    roll_dd     = rolling_drawdown(returns, window=63)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f"Rolling Metrics — {label}", fontsize=13)

    axes[0].plot(returns.index, (1 + returns).cumprod(), color='steelblue')
    axes[0].set_ylabel('Cumulative Return')
    axes[0].set_title('Equity Curve')
    axes[0].axhline(1.0, color='gray', linestyle='--', linewidth=0.8)

    axes[1].plot(returns.index, roll_sharpe, color='darkorange')
    axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[1].set_ylabel('Rolling Sharpe (63d)')
    axes[1].set_title('Rolling Sharpe Ratio')

    axes[2].fill_between(returns.index, roll_dd, 0, color='crimson', alpha=0.4)
    axes[2].set_ylabel('Rolling Max DD (63d)')
    axes[2].set_title('Rolling Maximum Drawdown')

    plt.tight_layout()
    fname = label.lower().replace(' ', '_') + '_rolling_metrics.png'
    plt.savefig(fname, dpi=120)
    plt.show()
    print(f"  Saved {fname}")

    return {'label': label, 'maxdd': maxdd, 'alpha': alpha, 'beta': beta_val,
            'vol': ann_vol, 'sharpe': sharpe, 'sortino': sortino}


In [ ]:
# ── Linear Regression Strategy ───────────────────────────────────────────
df_return_lr = pd.read_csv('total_asset_history.csv')
df_return_lr = df_return_lr.rename(columns={'Unnamed: 0': 'Date'})
df_return_lr['Date'] = pd.to_datetime(df_return_lr['Date'])
df_return_lr = df_return_lr.loc[df_return_lr['Date'] >= datetime.date(2016, 1, 4)]
df_return_lr = df_return_lr.set_index('Date')
df_return_lr.head()


In [ ]:
zero_rate = pd.read_csv('~/Downloads/feds200628.csv', skiprows=9)
zero_rate['Date'] = pd.to_datetime(zero_rate['Date'])
zero = (zero_rate[['Date', 'SVENY01']]
        .loc[lambda d: (d['Date'] >= datetime.date(2016, 1, 4)) &
                       (d['Date'] <= datetime.date(2019, 12, 31))]
        .set_index('Date'))
# Convert annual % to daily rate
zero['SVENY01'] = zero['SVENY01'] / 100.0 / 252.0
zero.head()


In [ ]:
df_lr = df_return_lr.pct_change().dropna()
quotes_lr = df_lr.join(zero, how='inner')

returns_lr   = quotes_lr['total assets'].values
benchmark_lr = quotes_lr['SVENY01'].values

results_lr = summarise_strategy('Linear Regression', returns_lr, benchmark_lr)


## Kalman Filter Strategy

In [ ]:
# ── Kalman Filter Strategy ───────────────────────────────────────────────
df_return_kf = pd.read_csv('6pair_kalman.csv')
df_return_kf = df_return_kf.rename(columns={'Unnamed: 0': 'Date', 'cum_returns': 'asset'})
df_return_kf['Date'] = pd.to_datetime(df_return_kf['Date'])
df_return_kf = df_return_kf.set_index('Date')
df_return_kf.head()


In [ ]:
df_kf = df_return_kf.pct_change().dropna()
quotes_kf = df_kf.join(zero, how='inner')

returns_kf   = quotes_kf['asset'].values
benchmark_kf = quotes_kf['SVENY01'].values

results_kf = summarise_strategy('Kalman Filter', returns_kf, benchmark_kf)


## Strategy Comparison

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────
metrics = ['maxdd', 'alpha', 'beta', 'vol', 'sharpe', 'sortino']
labels  = ['Max Drawdown', 'Alpha', 'Beta', 'Annual Vol', 'Sharpe', 'Sortino']

print("\n{:<20} {:>18} {:>18}".format('Metric', 'Linear Regression', 'Kalman Filter'))
print('-' * 58)
for m, label in zip(metrics, labels):
    lr_val = results_lr.get(m, float('nan'))
    kf_val = results_kf.get(m, float('nan'))
    if m in ('maxdd', 'alpha'):
        print("{:<20} {:>18.4%} {:>18.4%}".format(label, lr_val, kf_val))
    else:
        print("{:<20} {:>18.4f} {:>18.4f}".format(label, lr_val, kf_val))
